# C4: measure the KV cache speed curve

**Goal:** measure our own no-cache and cached generator on identical fixed-length workloads. C3 established correctness; this lesson measures time, not another HF parity comparison.

**Spec:** both modes return exactly prompt + requested new tokens. Warm up both, synchronize CUDA around complete generation calls, alternate trial order, retain raw timings and report min/median/max. Do not assume any minimum speedup.

Use a **fresh Colab T4**, not a runtime retaining C3's FP32 model copies. Save previous results before replacing it. Run every cell in order. Local checks are static only; **runtime validation is pending your run**.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


## 1. Test first, then load one FP16 model

Benchmark tests verify token-count accounting, timing formulas, warmup/order, invalid inputs, EOS restoration on failure, CUDA memory and plotting. C2/C3 tests protect the generator/cache contracts.

Tests run in a separate process, so their models are released before the benchmark. One two-GPU test can skip on the single T4. A failed test stops the notebook.

Load HF weights in FP16, copy them to our model, then free HF. Timings use **only our model**, with the same weights in both modes.

In [ ]:
import gc, warnings
from pathlib import Path
import torch
from IPython.display import Image, display
from bench.cache_curve import run_case
from bench.hf_baseline import make_prompt_ids
from tinyserve.basics import load_model, MODEL_ID
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.envinfo import collect_env, save_result
from tinyserve.plots import plot_cache_curve

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_c2_generation.py",
                           "tests/test_c3_cache.py", "tests/test_c4_bench.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float16)
our_model = LlamaForCausalLM.from_huggingface(hf_model)
del hf_model
gc.collect()
torch.cuda.empty_cache()
print("model:", type(our_model).__name__, "dtype:", next(our_model.parameters()).dtype)
print("context limit:", our_model.config.max_position_embeddings)


## 2. Count the work before measuring time

Let P be prompt tokens and N be new tokens.

```
no-cache forward sizes: P, P+1, P+2, ..., P+N-1
cached forward sizes:   P, 1,   1,   ..., 1

token rows per layer:
no cache = N*P + N*(N-1)/2
cached   = P + N - 1
```

For P=128/N=32: **4,592 versus 159 rows**. This is NOT a predicted 28.9x speedup. Large matrices use the GPU differently from one-token matrices. Cached decode still reads model weights and all visible K/V.

**Two sweeps:** vary prompt length at fixed N=32, then vary generation length at fixed P=128. N=1 is important: both modes prefill once, but cache allocates/writes storage without any later reuse. Cache may be slower there.

## 3. Make the benchmark fair

An eight-token EOS answer and a 32-token continuation are different workloads. With your approval, `run_case` temporarily disables EOS stopping, then restores the original setting in `finally`, even on failure. The model can still select the EOS ID; these are fixed-length synthetic runs, not answer-quality demos. Normal generation is unchanged.

Use the same exact-length filler prompt, model, dtype and budget for both modes. FP16 token choices can differ through rounding: record that observation, but require the same number of tokens. C3's FP32 parity remains the correctness evidence.

Warm up a full call per mode. Alternate order over three paired trials. Do not call `empty_cache()` between trials: that would introduce cold allocator behavior. Each cached call still creates fresh logical KV storage.

In [ ]:
PROMPT_LENGTHS = (16, 64, 128, 256, 512)
NEW_TOKENS = 32
FIXED_PROMPT_LENGTH = 128
GENERATION_LENGTHS = (1, 8, 16, 32)
TRIALS = 3
original_eos = our_model.config.eos_token_id
experiment_config = {
    "model_id": MODEL_ID, "dtype": "float16", "batch": 1,
    "prompt_lengths": list(PROMPT_LENGTHS), "fixed_new_tokens": NEW_TOKENS,
    "fixed_prompt_length": FIXED_PROMPT_LENGTH, "generation_lengths": list(GENERATION_LENGTHS),
    "trials": TRIALS, "warmup_runs_per_mode": 1, "eos_disabled_during_benchmark": True,
    "timing_scope": "public generate only; includes Python/cache allocation, excludes loading/tokenization/serving",
}
prompt_rows, generation_rows = [], []
figure_paths = []

def save_progress(status):
    return save_result("C4_cache_curve", {
        "status": status, "config": experiment_config,
        "prompt_sweep": prompt_rows, "generation_sweep": generation_rows,
        "figures": figure_paths,
        "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
    })

def report_case(row):
    plain = row["no_cache"]["total_ms"]
    cached = row["cached"]["total_ms"]
    ratio = row["speedup"]
    print(f"P={row['prompt_tokens']:3d} N={row['new_tokens']:2d} "
          f"no-cache ms={plain['min']:.1f}/{plain['median']:.1f}/{plain['max']:.1f} "
          f"cached ms={cached['min']:.1f}/{cached['median']:.1f}/{cached['max']:.1f} "
          f"speedup={ratio['min']:.2f}/{ratio['median']:.2f}/{ratio['max']:.2f}x "
          f"FP16 token IDs equal={row['token_ids_equal_in_all_trials']}")
    if not row["token_ids_equal_in_all_trials"]:
        warnings.warn("Observed FP16 token drift; exact token counts are still enforced. Inspect saved IDs.", RuntimeWarning)

print("Eight unique cases; three trials/mode/case plus warmup. Ranges are min/median/max.")


## 4. Prompt-length sweep

Hold new tokens at 32. As the prompt grows, no-cache recomputes more old token rows every step. Cache prefills those rows once, then uses one-token queries.

Measure full generation wall time, not GPU enqueue time. CUDA synchronization brackets each timed call. Model loading, tokenization, warmup and saving are outside the timer. Python validation, greedy sampling and cache allocation are inside it.

A result is saved after each point with `status='partial'`. This cell can take several minutes. An interrupted run is not a complete curve.

In [ ]:
prompt_rows.clear()
generation_rows.clear()
figure_paths.clear()
saved_path = save_progress("partial")
for prompt_length in PROMPT_LENGTHS:
    benchmark_ids = make_prompt_ids(tokenizer, prompt_length, 1, DEVICE)[0]
    row = run_case(our_model, benchmark_ids, new_tokens=NEW_TOKENS, trials=TRIALS)
    assert our_model.config.eos_token_id == original_eos
    prompt_rows.append(row)
    report_case(row)
    saved_path = save_progress("partial")
print("Checkpoint:", saved_path)


## 5. Generation-length sweep

Hold the prompt at 128 tokens. For one new token, there is no cache reuse. Longer generations can amortize prefill and benefit from skipping old-token work.

Reuse P=128/N=32 from the first sweep. The other three points are new measurements. The one-token point is **one-token generation time**, not serving TTFT: tokenization, queueing and network time are absent.

Do not estimate precise decode latency by subtracting separately measured runs. C4 reports complete-call time; serving TTFT and inter-token latency belong to Part E.

In [ ]:
assert len(prompt_rows) == len(PROMPT_LENGTHS), "Complete the prompt sweep first."
generation_rows.clear()
figure_paths.clear()
saved_path = save_progress("partial")
benchmark_ids = make_prompt_ids(tokenizer, FIXED_PROMPT_LENGTH, 1, DEVICE)[0]
for budget in GENERATION_LENGTHS:
    matching_rows = [row for row in prompt_rows
                     if row["prompt_tokens"] == FIXED_PROMPT_LENGTH and row["new_tokens"] == budget]
    row = matching_rows[0] if matching_rows else run_case(our_model, benchmark_ids, new_tokens=budget, trials=TRIALS)
    assert our_model.config.eos_token_id == original_eos
    generation_rows.append(row)
    report_case(row)
    saved_path = save_progress("partial")
print("Checkpoint:", saved_path)


## 6. Plot and interpret, without assuming the answer

Left: median full generation time; shaded bands are trial min/max. Right: paired no-cache time divided by cached time. Above 1 favors cache; below 1 means cached was slower. These bands are observed ranges, not confidence intervals.

Speedup is calculated within each trial pair, then summarized. Its median can differ from dividing the two latency medians.

Look for where cache starts helping, how the benefit changes with prompt length, and how N=1 differs. A regression is evidence to inspect, not something to hide. These curves describe this Python/PyTorch reference engine on this runtime.

In [ ]:
assert len(generation_rows) == len(GENERATION_LENGTHS), "Complete the generation sweep first."
image_dir = Path("docs") / "guide" / "img"
image_dir.mkdir(parents=True, exist_ok=True)
prompt_figure = image_dir / "C4_prompt_curve.png"
generation_figure = image_dir / "C4_generation_curve.png"
plot_cache_curve(prompt_rows, prompt_figure, x_key="prompt_tokens")
plot_cache_curve(generation_rows, generation_figure, x_key="new_tokens")
figure_paths[:] = [str(prompt_figure), str(generation_figure)]
display(Image(filename=str(prompt_figure)))
display(Image(filename=str(generation_figure)))


## 7. Tokens/s and memory: use the correct denominator

`generated tokens/s = N / total seconds`. Do not count P prompt tokens as generated output. This is end-to-end **within generate**, not server throughput.

Theoretical reserved K/V bytes are `(P+N) * 22 KiB` in FP16. The cache reserves capacity even though the final predicted token is not processed.

Measured CUDA peak includes weights and temporary tensors. Incremental peak subtracts memory allocated just before the call. It is not cache storage alone: caching adds K/V but can remove large full-prefix attention/logit temporaries. Neither measure is allocator-reserved memory or total process GPU usage.

In [ ]:
for row in prompt_rows:
    plain_rate = row["no_cache"]["end_to_end_tokens_per_s"]["median"]
    cached_rate = row["cached"]["end_to_end_tokens_per_s"]["median"]
    plain_peak = row["no_cache"]["incremental_peak_allocated_bytes"]["median"] / 2**20
    cached_peak = row["cached"]["incremental_peak_allocated_bytes"]["median"] / 2**20
    print(f"P={row['prompt_tokens']:3d}: generated tok/s no-cache={plain_rate:.2f}, cached={cached_rate:.2f}; "
          f"reserved K/V={row['cache_reserved_bytes'] / 2**20:.2f} MiB; "
          f"incremental peak no-cache={plain_peak:.2f}, cached={cached_peak:.2f} MiB")


## 8. Save the complete experiment and interview questions

1. **Why can cache lose at N=1?** Allocation and writes have no later forward on which to pay off.
2. **Why does fewer token rows not imply the same wall-time ratio?** Matrix efficiency, weight/KV traffic, prefill and Python overhead still matter.
3. **What makes the comparison fair?** Same model/dtype/prompt and exact budgets, warmup, CUDA synchronization, alternating order and raw trials.

Save the JSON, both PNGs and this notebook **with outputs**. Only the final save marks `status='complete'`. Share the JSON to record measured results; no speed claim is made in advance.

**Next after C4 validation: D1, flattened batching and attention metadata.**

In [ ]:
assert [row["prompt_tokens"] for row in prompt_rows] == list(PROMPT_LENGTHS)
assert [row["new_tokens"] for row in generation_rows] == list(GENERATION_LENGTHS)
assert all(row["new_tokens"] == NEW_TOKENS for row in prompt_rows)
assert all(row["prompt_tokens"] == FIXED_PROMPT_LENGTH for row in generation_rows)
assert len(figure_paths) == 2 and all(Path(path).is_file() for path in figure_paths)
saved_path = save_progress("complete")
print(saved_path)
print(saved_path.read_text())


In [ ]:
del our_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
